In [3]:
import pandas as pd
import numpy as np

# Load sales dataset
df = pd.read_csv("featured_sales.csv")

# Unique products
products = df["Product"].unique()

np.random.seed(42)

inventory = pd.DataFrame({
    "Product": products,
    "Current_Stock": np.random.randint(100, 600, len(products)),
    "Lead_Time": np.random.randint(2, 8, len(products)),     # Days
    "Safety_Stock": np.random.randint(40, 120, len(products))
})

inventory.to_csv("inventory.csv", index=False)

print(inventory)
print("\nInventory dataset created successfully.")

                 Product  Current_Stock  Lead_Time  Safety_Stock
0         50% Dark Bites            202          3            86
1         70% Dark Bites            535          5           101
2          85% Dark Bars            448          7            90
3        99% Dark & Pure            370          7            94
4            After Nines            206          3           103
5           Almond Choco            171          5            42
6    Baker's Choco Chips            288          6            90
7   Caramel Stuffed Bars            120          2            46
8   Choco Coated Almonds            202          5            60
9          Drinking Coco            221          3           112
10               Eclairs            566          7            78
11      Fruit & Nut Bars            314          6            57
12    Manuka Honey Choco            430          5            43
13             Milk Bars            558          2            99
14       Mint Chip Choco 

In [5]:
# Create Inventory Analysis
import pandas as pd

# Load files
sales = pd.read_csv("featured_sales.csv")
inventory = pd.read_csv("inventory.csv")

# Average daily demand
daily_demand = (
    sales.groupby("Product")["Boxes_Shipped"]
         .mean()
         .reset_index()
)

daily_demand.rename(
    columns={"Boxes_Shipped":"Average_Daily_Demand"},
    inplace=True
)

# Merge inventory with demand
inventory = inventory.merge(
    daily_demand,
    on="Product"
)

# Reorder Point
inventory["Reorder_Point"] = (
    inventory["Average_Daily_Demand"] *
    inventory["Lead_Time"]
) + inventory["Safety_Stock"]

# Inventory Status
def inventory_status(row):

    if row["Current_Stock"] < row["Reorder_Point"]:
        return "Reorder"

    elif row["Current_Stock"] > row["Reorder_Point"] * 2:
        return "Overstock"

    else:
        return "Healthy"

inventory["Status"] = inventory.apply(
    inventory_status,
    axis=1
)

# Recommended Order Quantity
inventory["Recommended_Order"] = (
    inventory["Reorder_Point"] -
    inventory["Current_Stock"]
)

inventory["Recommended_Order"] = (
    inventory["Recommended_Order"]
    .clip(lower=0)
    .round()
)

print(inventory)

                 Product  Current_Stock  Lead_Time  Safety_Stock  \
0         50% Dark Bites            202          3            86   
1         70% Dark Bites            535          5           101   
2          85% Dark Bars            448          7            90   
3        99% Dark & Pure            370          7            94   
4            After Nines            206          3           103   
5           Almond Choco            171          5            42   
6    Baker's Choco Chips            288          6            90   
7   Caramel Stuffed Bars            120          2            46   
8   Choco Coated Almonds            202          5            60   
9          Drinking Coco            221          3           112   
10               Eclairs            566          7            78   
11      Fruit & Nut Bars            314          6            57   
12    Manuka Honey Choco            430          5            43   
13             Milk Bars            558         

In [7]:
inventory.to_csv(
    "inventory_report.csv",
    index=False
)

print("\nInventory report saved successfully.")


Inventory report saved successfully.


In [8]:
print("\nInventory Summary")
print("-" * 40)

print("Total Products :", len(inventory))

print("Healthy Stock :", (inventory["Status"]=="Healthy").sum())

print("Need Reorder :", (inventory["Status"]=="Reorder").sum())

print("Overstock :", (inventory["Status"]=="Overstock").sum())


Inventory Summary
----------------------------------------
Total Products : 22
Healthy Stock : 1
Need Reorder : 21
Overstock : 0
